In [1]:
#!/usr/bin/env python3
"""
Replicate the Kalshi paper's Figure 3 (win percentage sorted by price) on NBA tip-off data.

Favorite-longshot bias shows up as the empirical curve sitting BELOW the 45-deg line at low
prices (longshots win less than their price implies) and ABOVE it at high prices.

  pip install pandas numpy matplotlib
  python nba_flb_graph.py
"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV = "nba_markets_enriched.csv"
OUT = "nba_flb_curve.png"
PRICE_MID, PRICE_LAST = "price_at_start_mid", "price_at_start_last"

df = pd.read_csv(CSV)
# clean: keep settled markets with a real 0/1 outcome and a usable tip-off price in (0,1)
df = df[df["result_binary"].isin([0, 1])].copy()
df["result_binary"] = df["result_binary"].astype(int)
df["price"] = df[PRICE_MID]
if PRICE_LAST in df.columns:
    df["price"] = df["price"].fillna(df[PRICE_LAST])   # fall back to last trade where no mid
df = df[df["price"].notna() & (df["price"] > 0) & (df["price"] < 1)]
print(f"{len(df)} usable settled markets")

# symmetrize: the paper's Fig 3 uses Yes AND No contracts. A yes-price P with outcome Y also
# implies a no-observation at price (1-P) with outcome (1-Y). This fills out the price axis so
# both tails are populated instead of only wherever yes-prices happened to land.
yes = pd.DataFrame({"price": df["price"],       "y": df["result_binary"]})
no  = pd.DataFrame({"price": 1 - df["price"],   "y": 1 - df["result_binary"]})
both = pd.concat([yes, no], ignore_index=True)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

# one point per integer cent (1..99), exactly like the paper's Figure 3
both["cents"] = (both["price"] * 100).round().clip(1, 99).astype(int)
rows = []
for cent, g in both.groupby("cents"):
    n, k = len(g), int(g["y"].sum())
    lo, hi = wilson(k, n)
    rows.append({"cents": cent, "n": n, "win_rate": k/n if n else np.nan, "lo": lo, "hi": hi})
cal = pd.DataFrame(rows).sort_values("cents")
print(cal.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

plt.figure(figsize=(7, 7))
plt.plot([0, 100], [0, 100], color="green", lw=1, label="45° (efficient: price = win rate)")
plt.fill_between(cal["cents"], cal["lo"]*100, cal["hi"]*100, color="gray", alpha=0.25, label="95% CI")
plt.plot(cal["cents"], cal["win_rate"]*100, color="crimson", lw=1, label="NBA empirical")
plt.xlabel("Price (cents)"); plt.ylabel("Win percentage")
plt.title("NBA Kalshi: win rate by price (favorite–longshot test)")
plt.xlim(0, 100); plt.ylim(0, 100); plt.legend(); plt.tight_layout()
plt.savefig(OUT, dpi=130)
print(f"saved {OUT}")

144747 usable settled markets
 cents     n  win_rate    lo    hi
     1  1914     0.005 0.003 0.010
     2   821     0.030 0.021 0.045
     3   599     0.027 0.017 0.043
     4  1852     0.033 0.026 0.043
     5   840     0.051 0.038 0.068
     6  2311     0.042 0.034 0.050
     7   996     0.059 0.046 0.076
     8  2308     0.075 0.065 0.087
     9   987     0.090 0.074 0.110
    10  2500     0.083 0.073 0.095
    11  1031     0.106 0.088 0.126
    12  2542     0.110 0.098 0.123
    13  1032     0.114 0.096 0.135
    14  2679     0.120 0.108 0.133
    15  1268     0.136 0.118 0.156
    16  2878     0.128 0.116 0.140
    17  1336     0.121 0.104 0.139
    18  3086     0.146 0.134 0.159
    19  1438     0.170 0.151 0.190
    20  2977     0.182 0.169 0.196
    21  1660     0.172 0.154 0.191
    22  3352     0.194 0.181 0.207
    23  1181     0.219 0.197 0.244
    24  3485     0.225 0.211 0.239
    25  1627     0.212 0.193 0.233
    26  4020     0.221 0.209 0.234
    27  1336     0.246 0.

In [2]:
dk = df[df['market_type'] == 'player_prop']
dk

,market_ticker,event_ticker,series_ticker,market_type,title,yes_sub_title,no_sub_title,result,result_binary,status,...,open_interest,tipoff_ts_used,tipoff_source,candle_ts,price_at_start_yes_bid,price_at_start_yes_ask,price_at_start_mid,price_at_start_last,n_candles_in_window,price
59279,KXNBAPTS-26JUN05NYKSAS-SASSCASTLE5-17,KXNBAPTS-26JUN05NYKSAS,KXNBAPTS,player_prop,Stephon Castle: 17+ points,Stephon Castle: 17+,Stephon Castle: 17+,no,0,finalized,...,13454.96,1780707600,expected_expiration-150m,1.780708e+09,0.40,0.46,0.430,0.50,98,0.430
59280,KXNBAPTS-26JUN05NYKSAS-NYJKTOWNS32-30,KXNBAPTS-26JUN05NYKSAS,KXNBAPTS,player_prop,Karl-Anthony Towns: 30+ points,Karl-Anthony Towns: 30+,Karl-Anthony Towns: 30+,no,0,finalized,...,19015.99,1780707600,expected_expiration-150m,1.780708e+09,0.03,0.04,0.035,NaN,67,0.035
59282,KXNBAPTS-26JUN05NYKSAS-NYKJALVARADO5-15,KXNBAPTS-26JUN05NYKSAS,KXNBAPTS,player_prop,Jose Alvarado: 15+ points,Jose Alvarado: 15+,Jose Alvarado: 15+,no,0,finalized,...,2329.94,1780707600,expected_expiration-150m,1.780708e+09,0.03,0.13,0.080,NaN,55,0.080
59286,KXNBAPTS-26JUN05NYKSAS-SASVWEMBANYAMA1-35,KXNBAPTS-26JUN05NYKSAS,KXNBAPTS,player_prop,Victor Wembanyama: 35+ points,Victor Wembanyama: 35+,Victor Wembanyama: 35+,no,0,finalized,...,66076.64,1780707600,expected_expiration-150m,1.780708e+09,0.13,0.14,0.135,0.14,103,0.135
59287,KXNBAPTS-26JUN05NYKSAS-SASVWEMBANYAMA1-30,KXNBAPTS-26JUN05NYKSAS,KXNBAPTS,player_prop,Victor Wembanyama: 30+ points,Victor Wembanyama: 30+,Victor Wembanyama: 30+,no,0,finalized,...,150741.51,1780707600,expected_expiration-150m,1.780708e+09,0.24,0.29,0.265,0.27,110,0.265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156106,KXNBA3D-25DEC11PORNOP-PORDAVDIJAPF,KXNBA3D-25DEC11PORNOP,KXNBA3D,player_prop,Portland at New Orleans: Triple Doubles: Deni ...,Deni Avdija,Deni Avdija,no,0,finalized,...,0.00,1765503000,expected_expiration-150m,1.765503e+09,0.16,0.27,0.215,NaN,51,0.215
156108,KXNBAH2HPRA-26JUN03JBRUNSON11VWEMBANYAMA1-JBRU...,KXNBAH2HPRA-26JUN03JBRUNSON11VWEMBANYAMA1,KXNBAH2HPRA,player_prop,Will Jalen Brunson record more combined points...,Jalen Brunson records more combined PRA than W...,Jalen Brunson records more combined PRA than W...,no,0,finalized,...,549.21,1780534800,expected_expiration-150m,1.780534e+09,0.01,0.60,0.305,NaN,21,0.305
156109,KXNBAH2HPRA-26JUN03JBRUNSON11VWEMBANYAMA1-VWEM...,KXNBAH2HPRA-26JUN03JBRUNSON11VWEMBANYAMA1,KXNBAH2HPRA,player_prop,Will Victor Wembanyama record more combined po...,Victor Wembanyama records more combined PRA th...,Victor Wembanyama records more combined PRA th...,yes,1,finalized,...,264.83,1780534800,expected_expiration-150m,1.780534e+09,0.40,0.98,0.690,NaN,20,0.690
156114,KXNBA3D-25DEC04MINNOP-MINAEDWARDSSG,KXNBA3D-25DEC04MINNOP,KXNBA3D,player_prop,Minnesota at New Orleans: Triple Doubles: Anth...,Anthony Edwards,Anthony Edwards,no,0,finalized,...,0.00,1764898200,expected_expiration-150m,1.764896e+09,0.00,0.96,0.480,NaN,6,0.480


In [3]:
#!/usr/bin/env python3

import numpy as np
import pandas as pd

TOP = 30
MIN_N = 10        # ignore cents with fewer than this many observations (raise to cut noise)
dk = dk.drop_duplicates(subset="market_ticker", keep="last")
dk = dk[dk["result_binary"].isin([0, 1])].copy()
dk["result_binary"] = dk["result_binary"].astype(int)
dk["price"] = dk["price_at_start_mid"]
if "price_at_start_last" in dk.columns:
    dk["price"] = dk["price"].fillna(dk["price_at_start_last"])
dk = dk[dk["price"].notna() & (dk["price"] > 0) & (dk["price"] < 1)]

# symmetrize (Yes + No), same basis as the curve. Each row is a real buyable side.
both = pd.concat([
    pd.DataFrame({"price": df["price"],     "y": df["result_binary"]}),
    pd.DataFrame({"price": 1 - df["price"], "y": 1 - df["result_binary"]}),
], ignore_index=True)
both["cents"] = (both["price"] * 100).round().clip(1, 99).astype(int)

g = both.groupby("cents")["y"].agg(n="size", empirical_prob="mean").reset_index()
g["kalshi_prob"] = g["cents"] / 100
g["diff"] = g["empirical_prob"] - g["kalshi_prob"]
se = np.sqrt(g["kalshi_prob"] * (1 - g["kalshi_prob"]) / g["n"])
g["z"] = g["diff"] / se
g["signif"] = g["z"].abs() > 1.96

out = (g[g["n"] >= MIN_N]
       .assign(abs_diff=g["diff"].abs())
       .sort_values("abs_diff", ascending=False)
       .head(TOP))

cols = ["cents", "kalshi_prob", "empirical_prob", "diff", "z", "signif", "n"]
idk = out[cols]
idk = idk.sort_values('diff', ascending=False)
idk

,cents,kalshi_prob,empirical_prob,diff,z,signif,n
59,60,0.60,0.724041,0.124041,21.316711,True,7088
62,63,0.63,0.695951,0.065951,4.942205,True,1309
54,55,0.55,0.612284,0.062284,8.994125,True,5161
52,53,0.53,0.590117,0.060117,5.985140,True,2469
69,70,0.70,0.757752,0.057752,8.253503,True,4289
60,61,0.61,0.666667,0.056667,4.315886,True,1380
82,83,0.83,0.886559,0.056559,4.876667,True,1049
50,51,0.51,0.565980,0.055980,8.328925,True,5532
70,71,0.71,0.765415,0.055415,4.587325,True,1411
61,62,0.62,0.668633,0.048633,6.521940,True,4237
